In [1]:
import os
import requests

In [2]:
from functions_panoramax_api import *

In [3]:
data = download_image("0e4308df-0f5e-45e2-84ef-4b350a7b33cb")

Directory verified: streetview_images
Fetching metadata from: https://api.panoramax.xyz/api/pictures/0e4308df-0f5e-45e2-84ef-4b350a7b33cb
Found HD image URL.
Image URL extracted: https://panoramax.openstreetmap.fr/images/0e/43/08/df/0f5e-45e2-84ef-4b350a7b33cb.jpg
An error occurred: [Errno 2] No such file or directory: ''


In [4]:
data

In [5]:
# LATITUDE, LONGITUDE = 49.1891701, -0.4072637
LATITUDE, LONGITUDE = 49.187321, -0.395511
RADIUS = 5.0

some_images = get_images_at(LATITUDE,LONGITUDE,RADIUS)

Searching area: -0.395579721820295,49.18727608423579,-0.395442278179705,49.187365915764204
Total images found: 2
360° images found: 2


In [6]:
some_images = get_images_at(LATITUDE,LONGITUDE,RADIUS)
some_images = down_load_images_from_features(some_images)

Searching area: -0.395579721820295,49.18727608423579,-0.395442278179705,49.187365915764204
Total images found: 2
360° images found: 2
Found HD image URL.
Image URL extracted: https://panoramax.openstreetmap.fr/images/82/09/d2/0d/e239-4d8a-83a8-de0408ce13c8.jpg
Success! Image saved to: streetview_images\8209d20d-e239-4d8a-83a8-de0408ce13c8.jpg
Found HD image URL.
Image URL extracted: https://panoramax.openstreetmap.fr/images/3f/42/d9/17/3a90-4e83-8a3f-c748c9ec09a7.jpg
Success! Image saved to: streetview_images\3f42d917-3a90-4e83-8a3f-c748c9ec09a7.jpg


In [7]:
print([[x.get('id'), x.get('distance_from_location')] for x in some_images])

[['8209d20d-e239-4d8a-83a8-de0408ce13c8', 5.499805399228335], ['3f42d917-3a90-4e83-8a3f-c748c9ec09a7', 6.028120847262711]]


In [18]:
found_svp = get_images_at(LATITUDE, LONGITUDE, RADIUS)

svp_dist = []
for x in found_svp:
    svp_dist.append(x.get('distance_from_location'))

print(min(svp_dist))
print(svp_dist.index(min(svp_dist)))
min_dist_idx = svp_dist.index(min(svp_dist))
svp = found_svp[min_dist_idx]
print(svp)
svp_id = found_svp[min_dist_idx].get('id')
print(svp_id)

Searching area: -0.395579721820295,49.18727608423579,-0.395442278179705,49.187365915764204
Total images found: 2
360° images found: 2
5.499805399228335
0
{'id': '8209d20d-e239-4d8a-83a8-de0408ce13c8', 'bbox': [-0.3955747, 49.1873476, -0.3955747, 49.1873476], 'type': 'Feature', 'links': [{'rel': 'root', 'href': 'https://api.panoramax.xyz/api/', 'type': 'application/json', 'title': 'Instance catalog'}, {'rel': 'parent', 'href': 'https://api.panoramax.xyz/api/collections/8482bcc7-e1f9-4bb0-be91-3208cf192969', 'type': 'application/json'}, {'rel': 'self', 'href': 'https://api.panoramax.xyz/api/collections/8482bcc7-e1f9-4bb0-be91-3208cf192969/items/8209d20d-e239-4d8a-83a8-de0408ce13c8', 'type': 'application/geo+json'}, {'rel': 'collection', 'href': 'https://api.panoramax.xyz/api/collections/8482bcc7-e1f9-4bb0-be91-3208cf192969', 'type': 'application/json'}, {'rel': 'license', 'href': 'https://creativecommons.org/licenses/by-sa/4.0/', 'title': 'License for this object (CC-BY-SA-4.0)'}, {'id':

In [2]:
import tkinter as tk
import tkintermapview
import requests

def load_panoramax_coverage(coords):
    lat, lon = coords
    print(f"Fetching Panoramax coverage near Lat: {lat}, Lon: {lon}...")
    
    # 1. Create a bounding box around the clicked area (roughly 2km radius)
    delta = 0.02 
    min_lon, min_lat = lon - delta, lat - delta
    max_lon, max_lat = lon + delta, lat + delta
    bbox = f"{min_lon},{min_lat},{max_lon},{max_lat}"
    
    # 2. Query the Panoramax API for up to 1000 image locations in this box
    api_url = f"https://api.panoramax.xyz/api/search?bbox={bbox}&limit=1000"
    
    try:
        response = requests.get(api_url).json()
        features = response.get("features", [])
        
        if not features:
            print("No coverage found in this immediate area.")
            return

        print(f"Found {len(features)} images. Drawing orange coverage lines...")

        # 3. Group the individual images by their "sequence" (collection)
        # This allows us to draw continuous lines down the streets
        sequences = {}
        for feature in features:
            # Panoramax stores the sequence ID in the 'collection' field
            seq_id = feature["collection"]
            
            # GeoJSON uses [Longitude, Latitude], but tkintermapview uses (Latitude, Longitude)
            img_lon, img_lat = feature["geometry"]["coordinates"]
            
            if seq_id not in sequences:
                sequences[seq_id] = []
            sequences[seq_id].append((img_lat, img_lon))
            
        # 4. Draw the sequences on the map
        for seq_id, coordinates in sequences.items():
            if len(coordinates) > 1:
                # If there are multiple points, draw a connecting orange line
                map_widget.set_path(coordinates, color="orange", width=4)
            else:
                # If it's a single isolated point, place an orange marker
                map_widget.set_marker(coordinates[0][0], coordinates[0][1], text="Point", marker_color_circle="orange", marker_color_outside="orange")
                
        print("Coverage drawn successfully!")
            
    except Exception as e:
        print(f"Could not connect to Panoramax: {e}")

# Create the main window
root = tk.Tk()
root.geometry("900x700")
root.title("Caen Map - Panoramax Coverage Viewer")

# Create the map widget
map_widget = tkintermapview.TkinterMapView(root, width=900, height=700, corner_radius=0)
map_widget.pack(fill="both", expand=True)

# Set base tile server to Standard OpenStreetMap
map_widget.set_tile_server("https://a.tile.openstreetmap.org/{z}/{x}/{y}.png", max_zoom=19)

# Set default location to Caen
map_widget.set_address("Caen, France")
map_widget.set_zoom(14)

# Add right-click command to load coverage
map_widget.add_right_click_menu_command(label="Load Panoramax Coverage Here", 
                                        command=load_panoramax_coverage, 
                                        pass_coords=True)

root.mainloop()

Status code 403 from https://nominatim.openstreetmap.org/search: ERROR - 403 Client Error: Forbidden for url: https://nominatim.openstreetmap.org/search?q=Caen%2C+France&format=jsonv2&addressdetails=1&limit=1
